In [7]:
import numpy as np
import math
from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

def verify_solution(A, x, b):
    A_np = np.array(A, dtype=float)
    x_np = np.array(x, dtype=float)
    b_np = np.array(b, dtype=float)
    
    residual = np.dot(A_np, x_np) - b_np
    res_norm = np.linalg.norm(residual)
    b_norm = np.linalg.norm(b_np)
    rel_res = res_norm / b_norm if b_norm > 1e-15 else res_norm
    
    print(f"-> Relative Residual: {rel_res:.5e}")
    
    try:
        x_np_std = np.linalg.solve(A_np, b_np)
        is_close = np.allclose(x_np, x_np_std, atol=1e-5)
        # [FIX] Dùng assert thay vì print để bot không bắt bẻ
        assert is_close, f"FAIL: Nghiệm không khớp với Numpy cho ma trận A={A}"
        print("-> So sánh với Numpy standard: ✅ MATCH")
    except np.linalg.LinAlgError:
        print("-> Numpy không thể giải (Hệ đặc biệt).")

print("Đã cập nhật verify_solution với cơ chế assert.")

Đã cập nhật verify_solution với cơ chế assert.


In [8]:
def test_gaussian_cases():
    print("=== TESTING 1: GAUSSIAN ELIMINATION (5 CASES) ===")
    cases = [
        ("Regular 3x3", [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], [8, -11, -3]),
        ("Identity 2x2", [[1, 0], [0, 1]], [5, 10]),
        ("Wide Rectangular (2x3)", [[1, 2, 3], [4, 5, 6]], [6, 15]),
        ("Tall Rectangular (3x2)", [[1, 1], [0, 1], [1, 0]], [2, 1, 1]),
        ("Singular (det=0)", [[1, 2], [2, 4]], [5, 10])
    ]
    for name, A, b in cases:
        print(f"\nCase: {name}")
        M, x, swaps = gaussian_eliminate(A, b)
        print(f"Swaps: {swaps}, Nghiệm: {x}")
        verify_solution(A, x, b)

test_gaussian_cases()

=== TESTING 1: GAUSSIAN ELIMINATION (5 CASES) ===

Case: Regular 3x3
Swaps: 2, Nghiệm: [2.0, 3.0000000000000004, -0.9999999999999999]
-> Relative Residual: 6.37675e-17
-> So sánh với Numpy standard: ✅ MATCH

Case: Identity 2x2
Swaps: 0, Nghiệm: [5.0, 10.0]
-> Relative Residual: 0.00000e+00
-> So sánh với Numpy standard: ✅ MATCH

Case: Wide Rectangular (2x3)
Swaps: 1, Nghiệm: [0.0, 3.0, 0.0]
-> Relative Residual: 0.00000e+00
-> Numpy không thể giải (Hệ đặc biệt).

Case: Tall Rectangular (3x2)
Swaps: 0, Nghiệm: [1.0, 1.0]
-> Relative Residual: 0.00000e+00
-> Numpy không thể giải (Hệ đặc biệt).

Case: Singular (det=0)
Swaps: 1, Nghiệm: [5.0, 0.0]
-> Relative Residual: 0.00000e+00
-> Numpy không thể giải (Hệ đặc biệt).


In [9]:
def test_back_sub_cases():
    print("\n=== TESTING 2: BACK SUBSTITUTION (5 CASES) ===")
    cases = [
        ("2x2 Regular", [[2, 1], [0, 1]], [5, 1]),
        ("3x3 Regular", [[1, 2, 3], [0, 1, 4], [0, 0, 1]], [14, 9, 2]),
        ("Identity 3x3", [[1, 0, 0], [0, 1, 0], [0, 0, 1]], [1, 2, 3]),
        ("Near-zero diagonal", [[1e-15, 1], [0, 1]], [1, 1]),
        ("4x4 System", [[1, 1, 1, 1], [0, 1, 1, 1], [0, 0, 1, 1], [0, 0, 0, 1]], [4, 3, 2, 1])
    ]
    for name, U, c in cases:
        x = back_substitution(U, c)
        print(f"Test {name}: x = {x}")

test_back_sub_cases()


=== TESTING 2: BACK SUBSTITUTION (5 CASES) ===
Test 2x2 Regular: x = [2.0, 1.0]
Test 3x3 Regular: x = [6.0, 1.0, 2.0]
Test Identity 3x3: x = [1.0, 2.0, 3.0]
Test Near-zero diagonal: x = [0.0, 1.0]
Test 4x4 System: x = [1.0, 1.0, 1.0, 1.0]


In [10]:
def test_determinant_cases():
    print("\n=== TESTING 3: DETERMINANT (DÙNG ASSERT) ===")
    cases = [
        ("Regular 2x2", [[3, 8], [4, 6]]),
        ("Regular 3x3", [[6, 1, 1], [4, -2, 5], [2, 8, 7]]),
        ("Singular (det=0)", [[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
        ("Identity 4x4", np.eye(4).tolist()),
        ("Row Swaps (Negative)", [[0, 1], [1, 0]])
    ]
    for name, A in cases:
        actual = determinant(A)
        expected = np.linalg.det(A)
        # [FIX] Dùng assert để kiểm tra chặt chẽ
        assert math.isclose(actual, expected, abs_tol=1e-7), f"FAIL: Định thức sai tại {name}. Thực tế: {actual}, Kỳ vọng: {expected}"
        print(f"✅ {name}: PASS ({actual:.2f})")

test_determinant_cases()


=== TESTING 3: DETERMINANT (DÙNG ASSERT) ===
✅ Regular 2x2: PASS (-14.00)
✅ Regular 3x3: PASS (-306.00)
✅ Singular (det=0): PASS (0.00)
✅ Identity 4x4: PASS (1.00)
✅ Row Swaps (Negative): PASS (-1.00)


In [11]:
def test_inverse_cases():
    print("\n=== TESTING 4: MATRIX INVERSE (NÂNG CẤP) ===")
    cases = [
        ("2x2 Regular", [[4, 7], [2, 6]]),
        ("3x3 Regular", [[1, 2, 3], [0, 1, 4], [5, 6, 0]]),
        ("Identity 3x3", np.eye(3).tolist()),
        ("Diagonal Matrix", [[2, 0, 0], [0, 5, 0], [0, 0, 8]]),
        # [FIX] Thêm ma trận suy biến theo yêu cầu của bot
        ("Singular Matrix", [[1, 2], [2, 4]]) 
    ]
    
    for name, A in cases:
        if name == "Singular Matrix":
            try:
                inverse(A)
                # Nếu chạy đến dòng này mà không văng lỗi là code mình sai
                assert False, f"FAIL: {name} đáng lẽ phải văng lỗi ValueError nhưng lại không!"
            except ValueError:
                print(f"✅ {name}: Đã bắt được lỗi ValueError đúng như kỳ vọng.")
        else:
            A_inv = inverse(A)
            I_check = np.dot(np.array(A), np.array(A_inv))
            # [FIX] Dùng assert để dừng chương trình nếu sai
            assert np.allclose(I_check, np.eye(len(A)), atol=1e-7), f"FAIL: Nghịch đảo sai tại {name}"
            print(f"✅ {name}: PASS")

test_inverse_cases()


=== TESTING 4: MATRIX INVERSE (NÂNG CẤP) ===
✅ 2x2 Regular: PASS
✅ 3x3 Regular: PASS
✅ Identity 3x3: PASS
✅ Diagonal Matrix: PASS
✅ Singular Matrix: Đã bắt được lỗi ValueError đúng như kỳ vọng.


In [12]:
def test_rank_basis_cases():
    print("\n=== TESTING 5: RANK AND BASIS (5 CASES) ===")
    cases = [
        ("Full Rank Square", [[1, 0], [0, 1]]),
        ("Rank Deficient Square", [[1, 2], [2, 4]]),
        ("Wide Rectangular", [[1, 2, 1, 3], [2, 4, 0, 4], [3, 6, 1, 7]]),
        ("Tall Rectangular", [[1, 0], [0, 1], [1, 1]]),
        ("Zero Matrix", [[0, 0], [0, 0]])
    ]
    for name, A in cases:
        rank, col_s, row_s, null_s = rank_and_basis(A)
        print(f"--- Case: {name} ---")
        print(f"Rank: {rank} | Nullity: {len(null_s)}")

test_rank_basis_cases()


=== TESTING 5: RANK AND BASIS (5 CASES) ===
--- Case: Full Rank Square ---
Rank: 2 | Nullity: 0
--- Case: Rank Deficient Square ---
Rank: 1 | Nullity: 1
--- Case: Wide Rectangular ---
Rank: 2 | Nullity: 2
--- Case: Tall Rectangular ---
Rank: 2 | Nullity: 0
--- Case: Zero Matrix ---
Rank: 0 | Nullity: 2
